# Multi Agent
## Supervisor Pattern
'중앙 관리자 에이전트가 하위 특화 에이전트에게 일을 시킴

> 관리자 Agent > 캘린더 agent / 이메일 agent

In [49]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### Calendar Agent

In [50]:
from langchain.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]


from datetime import date

CALENDAR_AGENT_PROMPT = (
    f"Today's date is {date.today().isoformat()}. "
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "If there is no suitable time slot, stop and confirm unavailability in your response. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

### Email Agent

In [51]:
@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"

EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)


In [52]:
@tool
def retrieve_member_info(sql_query: str, team:str = None) -> list[dict]:
    """
    주어진 PostgreSQL 쿼리를 실행하여 멤버(회원/직원) 정보를 데이터베이스에서 조회합니다.
    조회 결과로 id, name, email, team 정보를 포함하는 리스트를 반환합니다.
    team 을 입력할 경우, 해당 team 의 멤버들만 가져옵니다.
    """
    # 요청된 쿼리에 관계없이 가짜(Fake) 데이터를 반환하도록 구성된 목(Mock) 데이터입니다.
    # 기획팀, 영업팀, 개발팀 각각 3명씩 총 9명
    fake_db = [
        {"id": 1, "name": "김기획", "email": "plan1@example.com", "team": "기획팀"},
        {"id": 2, "name": "이기획", "email": "plan2@example.com", "team": "기획팀"},
        {"id": 3, "name": "박기획", "email": "plan3@example.com", "team": "기획팀"},
        {"id": 4, "name": "최영업", "email": "sales1@example.com", "team": "영업팀"},
        {"id": 5, "name": "정영업", "email": "sales2@example.com", "team": "영업팀"},
        {"id": 6, "name": "강영업", "email": "sales3@example.com", "team": "영업팀"},
        {"id": 7, "name": "조개발", "email": "dev1@example.com", "team": "개발팀"},
        {"id": 8, "name": "윤개발", "email": "dev2@example.com", "team": "개발팀"},
        {"id": 9, "name": "장개발", "email": "dev3@example.com", "team": "개발팀"},
    ]
    # 실제 환경이라면 여기서 sql_query를 DB에 날리겠지만, 여기서는 그대로 반환합니다.
    return list(filter(lambda m: m['team'] == team, fake_db)) if team else fake_db


MEMBER_AGENT_PROMPT = (
    "You are a database specialist assistant. "
    "Your task is to retrieve member/employee information from the database based on user requests. "
    "You must write and execute SQL queries using the `retrieve_member_info` tool.\n\n"
    "### Database Environment\n"
    "- SQL Dialect: PostgreSQL\n"
    "- Table Schema: `members` table has the following columns:\n"
    "  * id (SERIAL PRIMARY KEY)\n"
    "  * name (VARCHAR)\n"
    "  * email (VARCHAR)\n"
    "  * team (VARCHAR)\n\n"
    "### Constraints\n"
    "- STRICTLY READ-ONLY: You must ONLY execute READ (SELECT) operations. "
    "- Do NOT execute any INSERT, UPDATE, DELETE, or DROP queries under any circumstances.\n"
    "- Always summarize the retrieved data naturally in your final response."
)

In [53]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

supervisor_llm = init_chat_model('openai:gpt-5.4-mini')
worker_llm = init_chat_model('openai:gpt-4.1-mini')

calendar_agent = create_agent(
    model=worker_llm,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT
)

email_agent = create_agent(
    model=worker_llm,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT
)

member_agent = create_agent(
    model=worker_llm,
    tools=[retrieve_member_info],
    system_prompt=MEMBER_AGENT_PROMPT
)

In [ ]:
# from langchain.messages import HumanMessage

# result = calendar_agent.invoke({
#     'messages': [
#         HumanMessage('내일중에 비어있는 시간에 David과 놀러가기 잡아줘')
#     ]
# })



In [57]:
from langchain.messages import HumanMessage

@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [HumanMessage(request)]
    })

    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [HumanMessage(request)]
    })

    return result["messages"][-1].text


@tool 
def get_members_info(request:str) -> str:
    """Retrieve member or employee information using natural language.

    Use this when the user wants to search for team members, get employee contact details (email), 
    or query member lists by department/team. Handles SQL generation, member searching, and team filtering.

    Input: Natural language member search request (e.g., 'find emails of dev team members' or 'get info for planning team')
    """
    result = member_agent.invoke({
        'messages':[HumanMessage(request)]
    })

    return result["messages"][-1].content



In [58]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "You can get member info from SQL DB"
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence or in parallel as appropriate."
)

supervisor_agent = create_agent(
    model = supervisor_llm,
    tools = [schedule_event,manage_email,get_members_info],
    system_prompt = SUPERVISOR_PROMPT
)

In [56]:
supervisor_agent.invoke({'messages':[HumanMessage('김기획 email알려주삼')]})

{'messages': [HumanMessage(content='김기획 email알려주삼', additional_kwargs={}, response_metadata={}, id='9ba372d6-5a75-4a01-ac0f-ac86185477da'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 405, 'total_tokens': 431, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EOZHLKhNjcRU4YpYK9wgYUVW3WZfF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a7f4-2653-73e2-b7bd-d4d38621fe9d-0', tool_calls=[{'name': 'get_member_info', 'args': {'request': '김기획의 이메일 주소를 찾아줘'}, 'id': 'call_Adygk2Z